# 02 - GDP and Economic Growth Analysis

## Oman Economic Analysis Project

This notebook analyzes Oman's GDP trends, economic growth patterns, and compares with GCC peers.

### Key Questions
1. How has Oman's GDP evolved over time?
2. What is the GDP per capita trend?
3. How does Oman compare to other GCC countries?
4. What are the key growth periods and contractions?

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processor import DataProcessor
from src.visualizations import EconomicVisualizer

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)

%matplotlib inline

In [ ]:
# Load data
processor = DataProcessor(data_dir='../data')
viz = EconomicVisualizer(output_dir='../outputs/charts')

df = processor.load_data('gcc_economic_data.csv')
df = processor.clean_data(df)

print(f"Loaded {len(df):,} records")

## 1. Oman GDP Overview

In [ ]:
# GDP (current US$)
gdp_indicator = 'NY.GDP.MKTP.CD'

oman_gdp = df[(df['indicator_code'] == gdp_indicator) & 
              (df['country_code'] == 'OMN')].sort_values('year')

print("Oman GDP (current US$)")
print("=" * 50)

# Convert to billions
oman_gdp['gdp_billions'] = oman_gdp['value'] / 1e9

# Display recent years
recent = oman_gdp[oman_gdp['year'] >= 2015][['year', 'gdp_billions']].copy()
recent.columns = ['Year', 'GDP (Billion USD)']
print(recent.to_string(index=False))

In [ ]:
# Plot GDP trend
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(oman_gdp['year'], oman_gdp['gdp_billions'], 
        marker='o', linewidth=2, markersize=6, color='#E41A1C')

ax.fill_between(oman_gdp['year'], oman_gdp['gdp_billions'], alpha=0.3, color='#E41A1C')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP (Billion USD)', fontsize=12)
ax.set_title('Oman GDP (2000-2023)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add annotations for key events
ax.annotate('2008 Financial Crisis', xy=(2009, oman_gdp[oman_gdp['year']==2009]['gdp_billions'].values[0]),
            xytext=(2005, 70), arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')

ax.annotate('Oil Price Crash', xy=(2015, oman_gdp[oman_gdp['year']==2015]['gdp_billions'].values[0]),
            xytext=(2017, 90), arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('../outputs/charts/oman_gdp_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. GDP Growth Rate Analysis

In [ ]:
# GDP growth rate (annual %)
growth_indicator = 'NY.GDP.MKTP.KD.ZG'

oman_growth = df[(df['indicator_code'] == growth_indicator) & 
                 (df['country_code'] == 'OMN')].sort_values('year')

# Plot growth rate
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ca02c' if v >= 0 else '#d62728' for v in oman_growth['value']]
ax.bar(oman_growth['year'], oman_growth['value'], color=colors, edgecolor='white')

ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=oman_growth['value'].mean(), color='blue', linestyle='--', 
           label=f'Average: {oman_growth["value"].mean():.1f}%')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP Growth Rate (%)', fontsize=12)
ax.set_title('Oman GDP Growth Rate (Annual %)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/charts/oman_gdp_growth.png', dpi=150, bbox_inches='tight')
plt.show()

# Key statistics
print("\nGDP Growth Statistics")
print("=" * 40)
print(f"Average growth rate: {oman_growth['value'].mean():.2f}%")
print(f"Highest growth: {oman_growth['value'].max():.2f}% ({oman_growth.loc[oman_growth['value'].idxmax(), 'year']})")
print(f"Lowest growth: {oman_growth['value'].min():.2f}% ({oman_growth.loc[oman_growth['value'].idxmin(), 'year']})")

## 3. GDP Per Capita Analysis

In [ ]:
# GDP per capita
gdp_pc_indicator = 'NY.GDP.PCAP.CD'

oman_gdp_pc = df[(df['indicator_code'] == gdp_pc_indicator) & 
                 (df['country_code'] == 'OMN')].sort_values('year')

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(oman_gdp_pc['year'], oman_gdp_pc['value'], 
        marker='o', linewidth=2, markersize=6, color='#E41A1C')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP per Capita (USD)', fontsize=12)
ax.set_title('Oman GDP per Capita (2000-2023)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Format y-axis as currency
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../outputs/charts/oman_gdp_per_capita.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nLatest GDP per capita: ${oman_gdp_pc['value'].iloc[-1]:,.0f}")

## 4. GCC GDP Comparison

In [ ]:
# Compare GDP across GCC
gcc_gdp = df[df['indicator_code'] == gdp_indicator].copy()
gcc_gdp['gdp_billions'] = gcc_gdp['value'] / 1e9

# Latest year comparison
latest_year = gcc_gdp['year'].max()
gcc_latest = gcc_gdp[gcc_gdp['year'] == latest_year].sort_values('gdp_billions', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#E41A1C' if c == 'Oman' else '#377EB8' for c in gcc_latest['country_name']]
bars = ax.barh(gcc_latest['country_name'], gcc_latest['gdp_billions'], color=colors)

ax.set_xlabel('GDP (Billion USD)', fontsize=12)
ax.set_title(f'GCC GDP Comparison ({latest_year})', fontsize=14, fontweight='bold')

# Add value labels
for bar, value in zip(bars, gcc_latest['gdp_billions']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'${value:.0f}B', va='center', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_gdp_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# GDP trend comparison
fig, ax = plt.subplots(figsize=(14, 7))

country_colors = {
    'Oman': '#E41A1C',
    'Saudi Arabia': '#377EB8',
    'United Arab Emirates': '#4DAF4A',
    'Qatar': '#984EA3',
    'Kuwait': '#FF7F00',
    'Bahrain': '#A65628'
}

for country in gcc_gdp['country_name'].unique():
    country_data = gcc_gdp[gcc_gdp['country_name'] == country].sort_values('year')
    color = country_colors.get(country, '#999999')
    linewidth = 3 if country == 'Oman' else 1.5
    
    ax.plot(country_data['year'], country_data['gdp_billions'],
            marker='o', markersize=3, linewidth=linewidth,
            color=color, label=country)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP (Billion USD)', fontsize=12)
ax.set_title('GCC GDP Trends (2000-2023)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_gdp_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. GDP Per Capita Comparison

In [ ]:
# GDP per capita comparison
gcc_gdp_pc = df[df['indicator_code'] == gdp_pc_indicator].copy()
latest_gdp_pc = gcc_gdp_pc[gcc_gdp_pc['year'] == gcc_gdp_pc['year'].max()].sort_values('value', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#E41A1C' if c == 'Oman' else '#377EB8' for c in latest_gdp_pc['country_name']]
bars = ax.barh(latest_gdp_pc['country_name'], latest_gdp_pc['value'], color=colors)

ax.set_xlabel('GDP per Capita (USD)', fontsize=12)
ax.set_title(f'GCC GDP per Capita Comparison ({latest_gdp_pc["year"].iloc[0]})', 
             fontsize=14, fontweight='bold')

# Add value labels
for bar, value in zip(bars, latest_gdp_pc['value']):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'${value:,.0f}', va='center', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_gdp_per_capita_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Oman's ranking
oman_rank = latest_gdp_pc.reset_index(drop=True)
oman_rank['rank'] = range(len(oman_rank), 0, -1)
oman_position = oman_rank[oman_rank['country_name'] == 'Oman']['rank'].values[0]
print(f"\nOman's GDP per capita rank among GCC: {oman_position} of 6")

## 6. Key Findings Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS: Oman GDP Analysis")
print("=" * 60)

# Latest GDP
latest_gdp = oman_gdp['gdp_billions'].iloc[-1]
print(f"\n1. Current GDP: ${latest_gdp:.1f} billion USD")

# Growth
avg_growth = oman_growth['value'].mean()
print(f"\n2. Average GDP growth (2000-2023): {avg_growth:.1f}%")

# GDP per capita
latest_pc = oman_gdp_pc['value'].iloc[-1]
print(f"\n3. GDP per capita: ${latest_pc:,.0f} USD")

# GCC comparison
print(f"\n4. GCC ranking by GDP per capita: #{oman_position} of 6")

# Key observations
print("\n5. Key Observations:")
print("   - GDP growth highly correlated with oil prices")
print("   - 2015-2016 contraction due to oil price crash")
print("   - Recovery seen post-2020 pandemic")
print("   - Oman Vision 2040 driving diversification")

---

## Next: Trade Analysis

Continue to **03_trade_analysis.ipynb** for exports, imports, and trade balance analysis.